# Sand Dune TLS Data: TAM3C2 + 4D-OBC Analysis

Time-Adaptive M3C2 using the new `py4dgeo.tam3c2` module on a sand dune dataset.

**Dataset:** Sand Dune TLS Scans
- **Location:** `C:\rsa\research_proj\helios_simulation\output\Sand_Dune_tls`
- **Format:** XYZ files

**Workflow:**
1. Load epochs (timestamps parsed from filenames)
2. Sample corepoints from the reference epoch
3. Build a `TAM3C2` algorithm object and hand it to `SpatiotemporalAnalysis`
4. `analysis.add_epochs(*others)` triggers per-target time-adaptive M3C2
5. Run a custom 4D-OBC region-growing algorithm
6. Visualize aggregation diagnostics + extracted objects

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt

import py4dgeo
from py4dgeo import (
    TAM3C2,
    Weighting,
    read_epochs_from_folder,
    extract_reference_and_others,
    sample_corepoints,
)
from py4dgeo.segmentation import RegionGrowingSeed, temporal_averaging

## 1. Configuration

In [ ]:
data_path = r'C:\rsa\research_proj\helios_simulation\output\Sand_Dune_tls'

reference_timestamp = datetime(2020, 1, 1, 6, 0, 0)
max_epochs_after_reference = None  # Use all available epochs

# Corepoint sampling - adjust for sand dune scale
corepoint_voxel_size = 0.5

# TAM3C2 parameters - adjust for sand dune scale and dynamics
normal_radii = [0.5, 1.0]         # Smaller radii for finer features
max_window_ratio = [0.2, 0.4]
required_points = 10
cyl_radius = 0.5
max_distance = 2.0
registration_error = 0.01
sigma_ratio = 1.0
space_time_ratio = 1.0
weighting = Weighting.GAUSSIAN
keep_neighborhoods = False

# 4D-OBC parameters
obc_neighborhood_radius = 1.0
obc_min_segments = 10
obc_minperiod = 2
obc_height_threshold = 0.05
obc_thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
obc_smoothing_window = 3

## 2. Load epochs and filter time range

In [ ]:
# The default timestamp parser in read_epochs_from_folder expects YYMMDD.
# Our filenames are YYYYMMDD, so we need a custom parser.
import re
from py4dgeo.util import Py4DGeoError

def parse_sand_dune_timestamp(filename):
    m = re.search(r'(\d{8})_(\d{6})', os.path.basename(filename))
    if not m:
        return None
    try:
        return datetime.strptime(f"{m.group(1)}{m.group(2)}", '%Y%m%d%H%M%S')
    except (ValueError, IndexError):
        return None

def read_sand_dune_epochs(folder):
    files = sorted(f for f in os.listdir(folder) if f.lower().endswith('.xyz'))
    epochs = []
    for fn in files:
        ts = parse_sand_dune_timestamp(fn)
        if ts is None:
            print(f"Skipping (no timestamp in filename): {fn}")
            continue
        path = os.path.join(folder, fn)
        try:
            epoch = py4dgeo.read_from_xyz(path)
        except Exception as ex:
            print(f"Failed to read {fn}: {ex}")
            continue
        epoch.timestamp = ts
        epochs.append(epoch)
    return sorted(epochs, key=lambda e: e.timestamp)

epochs = read_sand_dune_epochs(data_path)
print(f"Loaded {len(epochs)} epochs")
if epochs:
    print(f"Date range: {epochs[0].timestamp}  ->  {epochs[-1].timestamp}")

In [ ]:
# Limit the time range used in the analysis
if max_epochs_after_reference is not None and epochs:
    sorted_eps = sorted(epochs, key=lambda e: e.timestamp)
    ref_idx = next((i for i, e in enumerate(sorted_eps) if e.timestamp == reference_timestamp), None)
    if ref_idx is None:
        raise ValueError(f"Reference {reference_timestamp} not in data")
    epochs = sorted_eps[:ref_idx + 1 + max_epochs_after_reference]
    print(f"Using {len(epochs)} epochs ({epochs[0].timestamp} -> {epochs[-1].timestamp})")

In [ ]:
reference_epoch, other_epochs = extract_reference_and_others(epochs, reference_timestamp)
print(f"Reference: {reference_epoch.timestamp}  ({len(reference_epoch.cloud):,} pts)")
print(f"Other epochs: {len(other_epochs)}")

corepoints = sample_corepoints(reference_epoch, method='voxel', voxel_size=corepoint_voxel_size)
print(f"Sampled {len(corepoints):,} corepoints  (voxel={corepoint_voxel_size} m)")

## 3. Build TAM3C2 and run the spatiotemporal analysis

In [ ]:
tam = TAM3C2(
    epochs_timeseries=epochs,
    max_window_ratio=max_window_ratio,
    normal_radii=normal_radii,
    required_points=required_points,
    weighting=weighting,
    sigma_ratio=sigma_ratio,
    space_time_ratio=space_time_ratio,
    keep_neighborhoods=keep_neighborhoods,
    corepoints=corepoints,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

output_path = os.path.join(os.getcwd(), 'sand_dune_tam3c2.zip')
analysis = py4dgeo.SpatiotemporalAnalysis(output_path, force=True)
analysis.reference_epoch = reference_epoch
analysis.corepoints = corepoints
analysis.m3c2 = tam

analysis.add_epochs(*other_epochs)
print(f"distances shape: {analysis.distances.shape}")
print(f"uncertainties shape: {analysis.uncertainties.shape}")

In [ ]:
# Temporal smoothing for 4D-OBC
analysis.smoothed_distances = temporal_averaging(
    analysis.distances, smoothing_window=obc_smoothing_window
)
print(f"smoothed shape: {analysis.smoothed_distances.shape}")

## 4. Aggregation diagnostics

In [ ]:
diag = tam.diagnostics()
for k, v in diag.items():
    if isinstance(v, np.ndarray):
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
    elif isinstance(v, list):
        print(f"  {k}: list of length {len(v)}")

# Save for later inspection
tam.save_diagnostics(os.path.join(os.getcwd(), 'sand_dune_tam3c2_diag.npz'))

In [ ]:
def plot_diag_map(values, title, cmap='viridis'):
    plt.figure(figsize=(10, 5))
    sc = plt.scatter(corepoints[:, 0], corepoints[:, 1], c=values, cmap=cmap, s=10)
    plt.colorbar(sc)
    plt.title(title)
    plt.xlabel('X [m]')
    plt.ylabel('Y [m]')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()

# Show aggregation behavior for the first target
if diag['target_timestamps']:
    tgt_col = 0
    nref = diag['n_before_ref'][:, tgt_col] + diag['n_after_ref'][:, tgt_col]
    ntgt = diag['n_before_tgt'][:, tgt_col] + diag['n_after_tgt'][:, tgt_col]

    plot_diag_map(nref, f"#aggregated REF epochs at target {diag['target_timestamps'][tgt_col]}")
    plot_diag_map(ntgt, f"#aggregated TGT epochs at target {diag['target_timestamps'][tgt_col]}")
    plot_diag_map(diag['n_points_ref'][:, tgt_col], 'Aggregated REF points (target 0)')

In [ ]:
if diag['target_timestamps']:
    mean_n_ref_per_target = (diag['n_before_ref'] + diag['n_after_ref']).mean(axis=0)
    mean_n_tgt_per_target = (diag['n_before_tgt'] + diag['n_after_tgt']).mean(axis=0)

    plt.figure(figsize=(11, 4))
    plt.plot(diag['target_timestamps'], mean_n_ref_per_target, label='mean # ref epochs aggregated')
    plt.plot(diag['target_timestamps'], mean_n_tgt_per_target, label='mean # tgt epochs aggregated')
    plt.xlabel('Target timestamp')
    plt.ylabel('Mean #epochs per corepoint')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 5. Data quality check

In [ ]:
sd = analysis.smoothed_distances
if sd is not None and sd.shape[1] > 0:
    valid_epochs_per_cp = np.sum(~np.isnan(sd), axis=1)
    max_abs_changes = np.nanmax(np.abs(sd), axis=1)

    has_enough = valid_epochs_per_cp >= obc_min_segments
    significant = max_abs_changes >= obc_height_threshold
    both = has_enough & significant

    print(f"Corepoints with >= {obc_min_segments} valid epochs: {has_enough.sum()} ({has_enough.mean()*100:.1f}%)")
    print(f"Corepoints with max |change| >= {obc_height_threshold} m: {significant.sum()} ({significant.mean()*100:.1f}%)")
    print(f"Meeting both: {both.sum()} ({both.mean()*100:.1f}%)")
    print(f"Max change: {np.nanmax(max_abs_changes):.4f} m   Mean change: {np.nanmean(max_abs_changes):.4f} m")
else:
    print("No distance data to analyze.")

## 6. Custom 4D-OBC seed algorithm (linear segments)

In [ ]:
class LinearChangeSeeds(py4dgeo.RegionGrowingAlgorithm):
    def find_seedpoints(self):
        from sklearn.tree import DecisionTreeRegressor
        from sklearn.linear_model import LinearRegression

        seeds = []
        min_magn = self.height_threshold
        minperiod = self.minperiod
        maxperiod = 12

        if self.seed_candidates is None:
            seed_candidates_curr = range(
                0, self.analysis.distances_for_compute.shape[0], self.seed_subsampling
            )
        else:
            seed_candidates_curr = self.seed_candidates[::self.seed_subsampling]

        def interp_nan(data):
            bad = np.isnan(data)
            n_nan = int(bad.sum())
            n_ok = len(data) - n_nan
            if n_ok > 3 and n_nan > 0:
                good = ~bad
                data[bad] = np.interp(bad.nonzero()[0], good.nonzero()[0], data[good])
            return data, n_nan, n_ok

        for cp_idx in seed_candidates_curr:
            ts = self.analysis.distances_for_compute[cp_idx, :].copy()
            ts, _, n_ok = interp_nan(ts)
            if n_ok <= 3:
                continue

            n_epochs = len(ts)
            xs = np.arange(n_epochs, dtype=float)
            dys = np.gradient(ts, xs)

            rgr = DecisionTreeRegressor(max_depth=4)
            rgr.fit(xs.reshape(-1, 1), dys.reshape(-1, 1))
            seg_id = rgr.predict(xs.reshape(-1, 1)).flatten()

            for y in np.unique(seg_id):
                msk = seg_id == y
                x0, x1 = int(round(xs[msk][0])), int(round(xs[msk][-1]))
                startp = max(x0 - 1, 0)
                stopp = min(x1 + 1, n_epochs - 1)
                if startp == 0 and stopp >= n_epochs - 1:
                    continue
                per = stopp - startp
                if per < minperiod or per > maxperiod:
                    continue
                if abs(ts[startp:stopp+1].max() - ts[startp:stopp+1].min()) < min_magn:
                    continue
                seeds.append(RegionGrowingSeed(cp_idx, startp, stopp))
        return seeds

## 7. Extract 4D-OBCs

In [ ]:
if analysis.smoothed_distances is not None and analysis.smoothed_distances.shape[1] > 0:
    max_changes = np.nanmax(np.abs(analysis.smoothed_distances), axis=1)
    valid_indices = np.where(~np.isnan(max_changes))[0]
    seed_candidates = list(valid_indices)
    print(f"Seed candidates: {len(seed_candidates)} / {len(max_changes)}")
else:
    seed_candidates = []
    print("No distances, no seed candidates.")

In [ ]:
if seed_candidates:
    algo = LinearChangeSeeds(
        neighborhood_radius=obc_neighborhood_radius,
        min_segments=obc_min_segments,
        minperiod=obc_minperiod,
        height_threshold=obc_height_threshold,
        thresholds=obc_thresholds,
        seed_candidates=seed_candidates,
        seed_subsampling=10,
    )

    analysis.invalidate_results(seeds=True, objects=True, smoothed_distances=False)
    objects = algo.run(analysis)
    print(f"Extracted {len(objects)} 4D-OBCs from {len(analysis.seeds)} seeds")
else:
    objects = []
    print("Skipping 4D-OBC extraction.")

## 8. Visualize results

In [ ]:
# Plotting the time series for the N-th corepoint and its seeds
if len(valid_indices) > 0:
    cp_idx_sel = valid_indices[0] # set copepoint index
else:
    print("No valid corepoints found for plotting.")
    cp_idx_sel = 0

timestamps = [e.timestamp for e in other_epochs][:analysis.smoothed_distances.shape[1]]
ts = analysis.smoothed_distances[cp_idx_sel]

plt.figure(figsize=(12, 5))
plt.plot(timestamps, ts, c='black', ls='--', lw=0.7, label='time series')

for sid, s in enumerate(s for s in analysis.seeds if s.index == cp_idx_sel):
    plt.plot(
        timestamps[s.start_epoch:s.end_epoch + 1],
        ts[s.start_epoch:s.end_epoch + 1],
        lw=2, label=f'seed {sid}: {s.start_epoch}-{s.end_epoch} epochs'
    )
plt.xlabel('Time'); plt.ylabel('Distance [m]')
plt.title(f'Corepoint {cp_idx_sel}')
plt.xticks(rotation=45)
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# plot the N-th object and its seed info
sel_object_idx = 0

if len(objects) > 0:
    sel_obj = analysis.objects[sel_object_idx]
    sel_seed = sel_obj.seed  # Use the seed stored on the object itself (NOT analysis.seeds[sel_object_idx])
    print(f"Object {sel_object_idx}: seed CP={sel_seed.index}, epochs {sel_seed.start_epoch}-{sel_seed.end_epoch}, size={len(sel_obj.indices)}")
    sel_obj.plot()

In [ ]:
# plot the N-th object and its seed info, with more details
from scipy.spatial import ConvexHull
from matplotlib.patches import Polygon
import matplotlib.colors as mcolors

sel_object_idx = 0 # object index to visualize

if len(objects) > 0:
    sel_object = analysis.objects[sel_object_idx]
    # get the seed from the object itself, NOT from analysis.seeds[i].
    sel_seed = sel_object.seed # seed index of the object
    seed_cp_idx = sel_seed.index # seed corepoint index

    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    ax1, ax2 = axs

    idxs = sel_object.indices
    epoch_of_interest = int(sel_object.end_epoch)
    magnitudes_of_interest = (analysis.smoothed_distances[:, epoch_of_interest] -
                              analysis.smoothed_distances[:, int(sel_object.start_epoch)])

    crange = 0.2
    cmap = plt.get_cmap('seismic_r').copy()
    norm = mcolors.CenteredNorm(halfrange=crange)
    cmapvals = norm(magnitudes_of_interest)

    for idx in idxs[::10]:
        ax1.plot(timestamps, analysis.smoothed_distances[idx],
                c=cmap(cmapvals[idx]), linewidth=0.5)
    ax1.plot(timestamps, analysis.smoothed_distances[seed_cp_idx],
            c='black', linewidth=1., label='Seed timeseries')
    ax1.axvspan(timestamps[sel_object.start_epoch], timestamps[sel_object.end_epoch],
               alpha=0.3, color='grey', label='4D-OBC timespan')
    ax1.legend()
    ax1.set_title('Time series of segmented 4D-OBC locations')
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Distance [m]')
    ax1.grid(True, alpha=0.3)
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

    cloud = analysis.corepoints.cloud
    subset_cloud = cloud[idxs, :2]

    d = ax2.scatter(cloud[:, 0], cloud[:, 1], c=magnitudes_of_interest,
                   cmap='seismic_r', vmin=-crange, vmax=crange, s=1)
    plt.colorbar(d, format='%.2f', label='Change magnitude [m]', ax=ax2)

    if len(subset_cloud) >= 3:
        hull = ConvexHull(subset_cloud)
        ax2.add_patch(Polygon(subset_cloud[hull.vertices, 0:2],
                             label='4D-OBC hull', fill=False, edgecolor='black'))

    ax2.scatter(cloud[seed_cp_idx, 0], cloud[seed_cp_idx, 1],
               marker='*', s=200, c='black',
               label=f'Seed (CP {seed_cp_idx})', zorder=5)

    ax2.set_title('Spatial distribution of 4D-OBC')
    ax2.set_xlabel('X [m]')
    ax2.set_ylabel('Y [m]')
    ax2.legend(loc='upper right')
    ax2.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    print("No objects to visualize!")